# ── CELL 1: Install & imports ─────────────────────────────────────────────────
## Dependency run

In [6]:
# ── CELL 1: Imports ───────────────────────────────────────────────────────────
import os, cv2, numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import ReduceLROnPlateau
import mediapipe as mp
from mediapipe.python.solutions import pose

# ✅ FIX 1: correct mediapipe alias
mp_pose = mp.solutions.pose

WINDOW    = 30
N_JOINTS  = 33
N_COORDS  = 3
INPUT_DIM = N_JOINTS * N_COORDS  # 99

print("Imports OK")
print(f"GPU available: {torch.cuda.is_available()}")

2026-03-14 14:19:32.282351: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773497972.304405     169 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773497972.311136     169 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773497972.329010     169 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773497972.329040     169 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773497972.329043     169 computation_placer.cc:177] computation placer alr

Imports OK
GPU available: True


In [7]:
# ── CELL 2: URFD extractor (image folders) ────────────────────────────────────
def extract_from_image_folder(folder_path, label, window=WINDOW):
    folder = Path(folder_path)
    frames = sorted(folder.glob('*.png'))
    if len(frames) < window:
        print(f"    [skip] {folder.name} — only {len(frames)} frames < {window}")
        return []

    frame_buffer = []
    # ✅ FIX 2: use mp_pose.Pose correctly
    with mp_pose.Pose(
        static_image_mode=True,   # True for image sequences (not video stream)
        model_complexity=1,
        min_detection_confidence=0.3
    ) as pose_model:
        for img_path in frames:
            img = cv2.imread(str(img_path))
            if img is None:
                frame_buffer.append(np.zeros(INPUT_DIM, dtype=np.float32))
                continue
            rgb    = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            result = pose_model.process(rgb)
            if result.pose_landmarks:
                kp = np.array(
                    [[lm.x, lm.y, lm.z]
                     for lm in result.pose_landmarks.landmark],
                    dtype=np.float32
                ).flatten()
            else:
                kp = np.zeros(INPUT_DIM, dtype=np.float32)
            frame_buffer.append(kp)

    # Count how many frames had valid pose
    nonzero = sum(1 for f in frame_buffer if np.any(f != 0))
    print(f"    {folder.name}: {len(frames)} frames, "
          f"pose detected in {nonzero} ({nonzero/len(frames)*100:.0f}%)")

    if nonzero < window:
        print(f"    [skip] Too few valid poses detected")
        return []

    sequences = []
    step = window // 2
    for i in range(0, len(frame_buffer) - window + 1, step):
        seq = np.stack(frame_buffer[i:i + window])
        sequences.append((seq.astype(np.float32), label))
    return sequences

In [9]:
# ── CELL 3: Build URFD ────────────────────────────────────────────────────────
URFD_PATH = "/kaggle/input/datasets/shahliza27/ur-fall-detection-dataset/UR_fall_detection_dataset_cam0_rgb"

# ✅ First verify the path exists and show structure
print("URFD root contents:")
for item in sorted(Path(URFD_PATH).iterdir())[:10]:
    print(f"  {item.name}  ({'dir' if item.is_dir() else 'file'})")

urfd_sequences = []

for seq_folder in sorted(Path(URFD_PATH).iterdir()):
    if not seq_folder.is_dir():
        continue
    name = seq_folder.name.lower()
    if 'cam0-rgb' not in name:
        continue
    label = 1 if name.startswith('fall') else 0
    seqs  = extract_from_image_folder(seq_folder, label)
    urfd_sequences.extend(seqs)

print(f"\n✅ URFD total sequences: {len(urfd_sequences)}")
fall_n = sum(1 for _, l in urfd_sequences if l == 1)
print(f"   Falls: {fall_n} | ADLs: {len(urfd_sequences) - fall_n}")

URFD root contents:
  adl-01-cam0-rgb  (dir)
  adl-02-cam0-rgb  (dir)
  adl-03-cam0-rgb  (dir)
  adl-04-cam0-rgb  (dir)
  adl-05-cam0-rgb  (dir)
  adl-06-cam0-rgb  (dir)
  adl-07-cam0-rgb  (dir)
  adl-08-cam0-rgb  (dir)
  adl-09-cam0-rgb  (dir)
  adl-10-cam0-rgb  (dir)


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1773498055.282892     264 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1773498055.341525     266 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1773498055.765389     265 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


    adl-01-cam0-rgb: 150 frames, pose detected in 136 (91%)


W0000 00:00:1773498064.509026     271 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1773498064.559393     271 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


    adl-02-cam0-rgb: 180 frames, pose detected in 141 (78%)


...

    fall-28-cam0-rgb: 66 frames, pose detected in 56 (85%)


W0000 00:00:1773498723.890238     586 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1773498723.929990     586 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


    fall-29-cam0-rgb: 99 frames, pose detected in 72 (73%)


W0000 00:00:1773498729.066938     591 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1773498729.109209     591 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


    fall-30-cam0-rgb: 70 frames, pose detected in 51 (73%)

✅ URFD total sequences: 700
   Falls: 157 | ADLs: 543


In [20]:
def extract_from_le2i_video(video_path, annotation_path, window=WINDOW):
    # ✅ Handle missing annotation — treat whole video as ADL
    fall_frames = set()
    if annotation_path is not None:
        fall_frames = parse_le2i_annotation(annotation_path)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"  [warn] Cannot open: {video_path}")
        return []

    frame_buffer, frame_indices = [], []
    pose_model = get_pose_model(static_mode=False)
    if pose_model is None:
        cap.release()
        return []

    try:
        idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            kp = extract_keypoints(frame, pose_model)
            frame_buffer.append(kp)
            frame_indices.append(idx)
            idx += 1
    finally:
        cap.release()
        pose_model.close()

    nonzero = sum(1 for f in frame_buffer if np.any(f != 0))
    print(f"  {Path(video_path).name}: {len(frame_buffer)} frames, "
          f"pose: {nonzero} ({nonzero/max(len(frame_buffer),1)*100:.0f}%), "
          f"fall frames annotated: {len(fall_frames)}")

    sequences = []
    step = window // 2
    for i in range(0, len(frame_buffer) - window + 1, step):
        seq           = np.stack(frame_buffer[i:i + window])
        window_frames = set(frame_indices[i:i + window])

        if fall_frames:
            overlap = len(window_frames & fall_frames) / window
            label   = 1 if overlap > 0.30 else 0
        else:
            label = 0   # no annotation = ADL

        sequences.append((seq.astype(np.float32), label))
    return sequences

In [22]:
def parse_le2i_annotation(ann_path):
    fall_frames = set()
    try:
        with open(ann_path, encoding='latin-1') as f:
            lines = f.read().strip().splitlines()

        nums = []
        for line in lines:
            parts = line.strip().split()
            for p in parts:
                if p.isdigit():
                    nums.append(int(p))
                    break

        if not nums:
            return fall_frames

        if len(nums) == 2 and nums[1] - nums[0] < 500:
            fall_frames = set(range(nums[0], nums[1] + 1))
        elif len(nums) == 1:
            fall_frames = set(range(max(0, nums[0]-15), nums[0]+30))
        else:
            fall_frames = set(nums)

        print(f"    [{Path(ann_path).name}] "
              f"{len(fall_frames)} fall frames, "
              f"range {min(fall_frames)}–{max(fall_frames)}")

    except Exception as e:
        print(f"  [warn] {ann_path}: {e}")
    return fall_frames

In [24]:
# ── COMPLETE CELL: All pose + extraction functions together ───────────────────
import mediapipe as mp
import cv2
import numpy as np
from pathlib import Path

mp_pose   = mp.solutions.pose
INPUT_DIM = 33 * 3  # 99
WINDOW    = 30

# ── Pose model factory ────────────────────────────────────────────────────────
def get_pose_model(static_mode=False):
    return mp_pose.Pose(
        static_image_mode=static_mode,
        model_complexity=1,
        min_detection_confidence=0.3,
        min_tracking_confidence=0.3
    )

# ── Single frame keypoint extractor ──────────────────────────────────────────
def extract_keypoints(frame_bgr, pose_model):
    rgb    = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    result = pose_model.process(rgb)
    if result.pose_landmarks:
        return np.array(
            [[lm.x, lm.y, lm.z]
             for lm in result.pose_landmarks.landmark],
            dtype=np.float32
        ).flatten()
    return np.zeros(INPUT_DIM, dtype=np.float32)

# ── Annotation parser ─────────────────────────────────────────────────────────
def parse_le2i_annotation(ann_path):
    fall_frames = set()
    try:
        with open(ann_path, encoding='latin-1') as f:
            lines = f.read().strip().splitlines()
        nums = []
        for line in lines:
            for p in line.strip().split():
                if p.isdigit():
                    nums.append(int(p))
                    break
        if not nums:
            return fall_frames
        if len(nums) == 2 and nums[1] - nums[0] < 500:
            fall_frames = set(range(nums[0], nums[1] + 1))
        elif len(nums) == 1:
            fall_frames = set(range(max(0, nums[0]-15), nums[0]+30))
        else:
            fall_frames = set(nums)
        print(f"    [{Path(ann_path).name}] "
              f"{len(fall_frames)} fall frames, "
              f"range {min(fall_frames)}–{max(fall_frames)}")
    except Exception as e:
        print(f"  [warn] {ann_path}: {e}")
    return fall_frames

# ── URFD: image folder extractor ──────────────────────────────────────────────
def extract_from_image_folder(folder_path, label, window=WINDOW):
    folder = Path(folder_path)
    frames = sorted(folder.glob('*.png'))
    if len(frames) < window:
        return []

    frame_buffer = []
    pose_model   = get_pose_model(static_mode=True)
    try:
        for img_path in frames:
            img = cv2.imread(str(img_path))
            if img is None:
                frame_buffer.append(np.zeros(INPUT_DIM, dtype=np.float32))
                continue
            frame_buffer.append(extract_keypoints(img, pose_model))
    finally:
        pose_model.close()

    nonzero = sum(1 for f in frame_buffer if np.any(f != 0))
    print(f"  {folder.name}: {len(frames)} frames, "
          f"pose: {nonzero} ({nonzero/len(frames)*100:.0f}%)")

    sequences, step = [], window // 2
    for i in range(0, len(frame_buffer) - window + 1, step):
        seq = np.stack(frame_buffer[i:i + window])
        sequences.append((seq.astype(np.float32), label))
    return sequences

# ── Le2i: video extractor ─────────────────────────────────────────────────────
def extract_from_le2i_video(video_path, annotation_path, window=WINDOW):
    fall_frames = set()
    if annotation_path is not None:
        fall_frames = parse_le2i_annotation(annotation_path)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"  [warn] Cannot open: {video_path}")
        return []

    frame_buffer, frame_indices = [], []
    pose_model = get_pose_model(static_mode=False)
    try:
        idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame_buffer.append(extract_keypoints(frame, pose_model))
            frame_indices.append(idx)
            idx += 1
    finally:
        cap.release()
        pose_model.close()

    nonzero = sum(1 for f in frame_buffer if np.any(f != 0))
    print(f"  {Path(video_path).name}: {len(frame_buffer)} frames, "
          f"pose: {nonzero} ({nonzero/max(len(frame_buffer),1)*100:.0f}%), "
          f"fall frames: {len(fall_frames)}")

    sequences, step = [], window // 2
    for i in range(0, len(frame_buffer) - window + 1, step):
        seq           = np.stack(frame_buffer[i:i + window])
        window_frames = set(frame_indices[i:i + window])
        overlap       = len(window_frames & fall_frames) / window if fall_frames else 0
        label         = 1 if overlap > 0.30 else 0
        sequences.append((seq.astype(np.float32), label))
    return sequences

print("✅ All functions defined")

# ── Quick smoke test ──────────────────────────────────────────────────────────
test_model = get_pose_model(static_mode=True)
blank      = np.zeros((480, 640, 3), dtype=np.uint8)
kp         = extract_keypoints(blank, test_model)
test_model.close()
print(f"✅ Pose extractor OK — output shape: {kp.shape}")  # should be (99,)

✅ All functions defined
✅ Pose extractor OK — output shape: (99,)


W0000 00:00:1773500348.838135     638 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1773500348.881408     638 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [25]:
# ── CORRECTED Le2i builder ────────────────────────────────────────────────────
LE2I_PATH = "/kaggle/input/datasets/tuyenldvn/falldataset-imvia"

le2i_sequences = []

# Exact structure per scene based on real paths:
# scene_key : (outer_folder, inner_folder, videos_subfolder, annotations_subfolder)
SCENE_MAP = {
    "Coffee_room_01": ("Coffee_room_01", "Coffee_room_01", "Videos",       "Annotation_files"),
    "Coffee_room_02": ("Coffee_room_02", "Coffee_room_02", "Videos",       "Annotations_files"),  # note the 's'
    "Home_01":        ("Home_01",        "Home_01",        "Videos",       "Annotation_files"),
    "Home_02":        ("Home_02",        "Home_02",        "Videos",       "Annotation_files"),
    "Lecture_room":   ("Lecture_room",   "Lecture room",   None,           None),   # flat — no subfolders
    "Office":         ("Office",         "Office",         None,           None),   # flat — videos directly inside
}

for scene_key, (outer, inner, vid_sub, ann_sub) in SCENE_MAP.items():
    base = Path(LE2I_PATH) / outer / inner

    if not base.exists():
        print(f"\n[skip] {scene_key} — base path not found: {base}")
        continue

    # Get video files
    if vid_sub:
        vid_dir = base / vid_sub
    else:
        vid_dir = base   # flat structure

    # Get annotation dir
    if ann_sub:
        ann_dir = base / ann_sub
    else:
        ann_dir = base   # flat structure

    video_files = sorted(vid_dir.glob('*.avi')) + sorted(vid_dir.glob('*.mp4'))
    print(f"\n{scene_key}: {len(video_files)} videos  |  vid_dir={vid_dir}")

    if not video_files:
        print(f"  [warn] No video files found in {vid_dir}")
        continue

    for vpath in video_files:
        # Match annotation by stem — e.g. "video (1)" → "video (1).txt"
        ann_path = ann_dir / (vpath.stem + '.txt')

        if not ann_path.exists():
            # Try searching for any matching file
            matches  = list(ann_dir.glob(f'{vpath.stem}*'))
            ann_path = matches[0] if matches else None

        if ann_path is None:
            print(f"  [warn] No annotation for {vpath.name} — labeling as ADL")
            # Videos without annotations in Le2i are typically ADL
            seqs = extract_from_le2i_video(str(vpath), None)
        else:
            seqs = extract_from_le2i_video(str(vpath), str(ann_path))

        le2i_sequences.extend(seqs)

    scene_falls = sum(1 for _, l in le2i_sequences if l == 1)
    print(f"  Running total — Falls: {scene_falls} | "
          f"ADLs: {len(le2i_sequences) - scene_falls}")

print(f"\n✅ Le2i total: {len(le2i_sequences)}")
fall_n = sum(1 for _, l in le2i_sequences if l == 1)
print(f"   Falls: {fall_n} | ADLs: {len(le2i_sequences) - fall_n}")


Coffee_room_01: 48 videos  |  vid_dir=/kaggle/input/datasets/tuyenldvn/falldataset-imvia/Coffee_room_01/Coffee_room_01/Videos
    [video (1).txt] 33 fall frames, range 48–80


[mp3float @ 0x44418a40] Header missing
[mp3float @ 0x44418a40] Header missing
[mp3float @ 0x44418a40] Header missing
W0000 00:00:1773500353.519587     641 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1773500353.567047     641 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


  video (1).avi: 157 frames, pose: 141 (90%), fall frames: 33
    [video (10).txt] 28 fall frames, range 211–238


[mp3float @ 0x44418a40] Header missing
W0000 00:00:1773500357.757860     646 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1773500357.794672     645 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


  video (10).avi: 362 frames, pose: 315 (87%), fall frames: 28
    [video (11).txt] 30 fall frames, range 375–404


...

  video (7).avi: 217 frames, pose: 217 (100%), fall frames: 0
  [warn] /kaggle/input/datasets/tuyenldvn/falldataset-imvia/Office/Office/video (8).avi: invalid literal for int() with base 10: '³'


W0000 00:00:1773502421.167061    1414 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1773502421.201226    1414 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


  video (8).avi: 419 frames, pose: 337 (80%), fall frames: 0
  [warn] /kaggle/input/datasets/tuyenldvn/falldataset-imvia/Office/Office/video (9).avi: invalid literal for int() with base 10: '²'


W0000 00:00:1773502432.167141    1417 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1773502432.198409    1417 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


  video (9).avi: 206 frames, pose: 173 (84%), fall frames: 0
  Running total — Falls: 223 | ADLs: 4558

✅ Le2i total: 4781
   Falls: 223 | ADLs: 4558


In [26]:
# # ── FIND EXACT KAGGLE PATH ────────────────────────────────────────────────────
# import os

# for root, dirs, files in os.walk("/kaggle/input"):
#     level = root.replace("/kaggle/input", '').count(os.sep)
#     if level > 6:
#         continue
#     indent = '  ' * level
#     folder_name = os.path.basename(root)
#     print(f"{indent}{root}/  ({len(files)} files)")
#     # Show first 3 files at each level
#     for f in sorted(files)[:3]:
#         print(f"{indent}  📄 {f}")

#  Merge, normalize, split

In [27]:
# ── CELL 6: Merge + sanity check ─────────────────────────────────────────────
all_seqs = urfd_sequences + le2i_sequences

print(f"URFD sequences  : {len(urfd_sequences)}")
print(f"Le2i sequences  : {len(le2i_sequences)}")
print(f"Total combined  : {len(all_seqs)}")

# ✅ Abort early if dataset is too small — saves wasting training time
if len(all_seqs) < 200:
    raise ValueError(
        f"❌ Only {len(all_seqs)} sequences — extraction failed. "
        "Check paths and pose detection output above before continuing."
    )

np.random.shuffle(all_seqs)
X = np.array([s[0] for s in all_seqs], dtype=np.float32)
y = np.array([s[1] for s in all_seqs], dtype=np.int64)

counts = np.bincount(y)
print(f"\nLabel 0 (ADL) : {counts[0]}  ({counts[0]/len(X)*100:.1f}%)")
print(f"Label 1 (FALL): {counts[1]}  ({counts[1]/len(X)*100:.1f}%)")
print(f"Imbalance ratio: {counts[0]/counts[1]:.1f}:1")
print(f"Tensor shape   : {X.shape}")   # should be (N, 30, 99) with N >> 500

URFD sequences  : 700
Le2i sequences  : 4781
Total combined  : 5481

Label 0 (ADL) : 5101  (93.1%)
Label 1 (FALL): 380  (6.9%)
Imbalance ratio: 13.4:1
Tensor shape   : (5481, 30, 99)


In [28]:

# ── CELL 8: Clean, normalize, split ──────────────────────────────────────────

# Re-shuffle at array level to break sliding-window locality
shuffle_idx = np.random.permutation(len(X))
X = X[shuffle_idx]
y = y[shuffle_idx]

# Remove sequences where pose was never detected (all zeros)
valid_mask = ~np.all(X.reshape(len(X), -1) == 0, axis=1)
removed = (~valid_mask).sum()
X = X[valid_mask]
y = y[valid_mask]
print(f"Removed {removed} all-zero sequences. Remaining: {len(X)}")

# Normalize per joint-coordinate across entire dataset
mean = X.mean(axis=(0, 1))   # (99,)
std  = X.std(axis=(0, 1)) + 1e-8
X    = (X - mean) / std

# Save — MUST match these exact values in vision_engine.py at inference
np.save('/kaggle/working/norm_mean.npy', mean)
np.save('/kaggle/working/norm_std.npy',  std)
print(f"Normalization stats saved. Mean range: [{mean.min():.3f}, {mean.max():.3f}]")

# 80/10/10 stratified split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

print(f"\nTrain : {X_train.shape} | fall%: {y_train.mean()*100:.1f}%")
print(f"Val   : {X_val.shape}   | fall%: {y_val.mean()*100:.1f}%")
print(f"Test  : {X_test.shape}  | fall%: {y_test.mean()*100:.1f}%")

Removed 400 all-zero sequences. Remaining: 5081
Normalization stats saved. Mean range: [-0.140, 0.717]

Train : (4064, 30, 99) | fall%: 7.5%
Val   : (508, 30, 99)   | fall%: 7.5%
Test  : (509, 30, 99)  | fall%: 7.5%


# PyTorch Dataset & DataLoader

In [32]:
# ── CELL: PyTorch Dataset & DataLoader ───────────────────────────────────────
class FallDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

class_counts    = np.bincount(y_train)
weights         = 1.0 / class_counts
sample_weights  = weights[y_train]
sampler         = WeightedRandomSampler(sample_weights, len(sample_weights))

train_loader = DataLoader(FallDataset(X_train, y_train),
                          batch_size=64, sampler=sampler)
val_loader   = DataLoader(FallDataset(X_val,   y_val),
                          batch_size=64, shuffle=False)
test_loader  = DataLoader(FallDataset(X_test,  y_test),
                          batch_size=64, shuffle=False)

print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")
print(f"Test  batches : {len(test_loader)}")

Train batches : 64
Val   batches : 8
Test  batches : 8


# 2-Layer Bidirectional LSTM model

In [33]:
# ── CELL: Model definition ────────────────────────────────────────────────────
class FallLSTM(nn.Module):
    """
    2-layer Bidirectional LSTM + attention for fall detection.
    Input  : (batch, 30, 99)  — 30 frames × 33 joints × xyz
    Output : (batch, 2)       — fall / no-fall logits
    """
    def __init__(self, input_dim=99, hidden_dim=128,
                 num_layers=2, num_classes=2, dropout=0.4):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size  = input_dim,
            hidden_size = hidden_dim,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout,
            bidirectional = True        # output dim = hidden_dim * 2 = 256
        )

        # Soft attention over time steps
        self.attention = nn.Linear(hidden_dim * 2, 1)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)                          # (B, T, 256)
        attn_w      = torch.softmax(
                          self.attention(lstm_out), dim=1)  # (B, T, 1)
        context     = (lstm_out * attn_w).sum(dim=1)        # (B, 256)
        return self.classifier(context)                     # (B, 2)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = FallLSTM().to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Device     : {device}")
print(f"Parameters : {total_params:,}")
print(f"Model      :\n{model}")

Device     : cuda
Parameters : 646,595
Model      :
FallLSTM(
  (lstm): LSTM(99, 128, num_layers=2, batch_first=True, dropout=0.4, bidirectional=True)
  (attention): Linear(in_features=256, out_features=1, bias=True)
  (classifier): Sequential(
    (0): Linear(in_features=256, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.4, inplace=False)
    (3): Linear(in_features=64, out_features=2, bias=True)
  )
)


# Training loop

In [35]:
# ── CELL: Training loop ───────────────────────────────────────────────────────
from sklearn.metrics import f1_score, recall_score, precision_score

# Weighted loss — penalise missed falls more than false alarms
# For patient monitoring: missing a fall is worse than false alarm
fall_weight = class_counts[0] / class_counts[1]   # ADL_count / FALL_count
print(f"Fall class weight: {fall_weight:.2f}x")

criterion = nn.CrossEntropyLoss(
    weight=torch.FloatTensor([1.0, fall_weight]).to(device)
)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=1e-3, weight_decay=1e-4
)
scheduler = ReduceLROnPlateau(
    optimizer, mode='max', patience=6,
    factor=0.5
)

EPOCHS        = 60
best_val_f1   = 0.0
history       = []

for epoch in range(EPOCHS):

    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    train_loss, train_preds, train_true = 0.0, [], []

    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        logits = model(X_b)
        loss   = criterion(logits, y_b)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item()
        train_preds.extend(logits.argmax(1).cpu().numpy())
        train_true.extend(y_b.cpu().numpy())

    # ── Validate ───────────────────────────────────────────────────────────
    model.eval()
    val_preds, val_true = [], []

    with torch.no_grad():
        for X_b, y_b in val_loader:
            logits = model(X_b.to(device))
            val_preds.extend(logits.argmax(1).cpu().numpy())
            val_true.extend(y_b.numpy())

    # ── Metrics ────────────────────────────────────────────────────────────
    train_acc    = np.mean(np.array(train_preds) == np.array(train_true))
    val_acc      = np.mean(np.array(val_preds)   == np.array(val_true))
    val_f1       = f1_score(val_true, val_preds, pos_label=1, zero_division=0)
    val_recall   = recall_score(val_true, val_preds, pos_label=1, zero_division=0)
    val_precision= precision_score(val_true, val_preds, pos_label=1, zero_division=0)
    avg_loss     = train_loss / len(train_loader)

    history.append({
        'epoch'     : epoch,
        'train_acc' : train_acc,
        'val_acc'   : val_acc,
        'val_f1'    : val_f1,
        'val_recall': val_recall,
        'loss'      : avg_loss,
    })

    # Save best model by F1 (better than accuracy for imbalanced data)
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), '/kaggle/working/fall_lstm_best.pt')

    scheduler.step(val_f1)

    if epoch % 5 == 0:
        print(f"Epoch {epoch:3d} | "
              f"Loss {avg_loss:.4f} | "
              f"Train {train_acc:.3f} | "
              f"Val Acc {val_acc:.3f} | "
              f"F1 {val_f1:.3f} | "
              f"Recall {val_recall:.3f} | "
              f"Precision {val_precision:.3f}"
              f"{'  ← best' if val_f1 == best_val_f1 else ''}")

print(f"\n✅ Training done. Best Val F1: {best_val_f1:.3f}")

Fall class weight: 12.37x
Epoch   0 | Loss 0.2304 | Train 0.675 | Val Acc 0.717 | F1 0.339 | Recall 0.974 | Precision 0.206  ← best
Epoch   5 | Loss 0.0485 | Train 0.934 | Val Acc 0.864 | F1 0.504 | Recall 0.921 | Precision 0.347
Epoch  10 | Loss 0.0250 | Train 0.967 | Val Acc 0.929 | F1 0.654 | Recall 0.895 | Precision 0.515
Epoch  15 | Loss 0.0206 | Train 0.973 | Val Acc 0.919 | F1 0.624 | Recall 0.895 | Precision 0.479
Epoch  20 | Loss 0.0167 | Train 0.976 | Val Acc 0.953 | F1 0.745 | Recall 0.921 | Precision 0.625
Epoch  25 | Loss 0.0126 | Train 0.977 | Val Acc 0.961 | F1 0.767 | Recall 0.868 | Precision 0.688
Epoch  30 | Loss 0.0055 | Train 0.992 | Val Acc 0.967 | F1 0.790 | Recall 0.842 | Precision 0.744
Epoch  35 | Loss 0.0060 | Train 0.992 | Val Acc 0.974 | F1 0.835 | Recall 0.868 | Precision 0.805  ← best
Epoch  40 | Loss 0.0053 | Train 0.992 | Val Acc 0.969 | F1 0.795 | Recall 0.816 | Precision 0.775
Epoch  45 | Loss 0.0045 | Train 0.994 | Val Acc 0.969 | F1 0.789 | Recall 0.

# Evaluation

In [36]:
# ── CELL: Final evaluation on test set ───────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix

# Load best checkpoint
model.load_state_dict(torch.load('/kaggle/working/fall_lstm_best.pt'))
model.eval()

test_preds, test_true = [], []
with torch.no_grad():
    for X_b, y_b in test_loader:
        logits = model(X_b.to(device))
        test_preds.extend(logits.argmax(1).cpu().numpy())
        test_true.extend(y_b.numpy())

print("=== TEST SET RESULTS ===\n")
print(classification_report(
    test_true, test_preds,
    target_names=['ADL (no fall)', 'FALL'],
    digits=3
))

cm = confusion_matrix(test_true, test_preds)
tn, fp, fn, tp = cm.ravel()
print(f"Confusion matrix:")
print(f"  True  ADL  predicted ADL  : {tn}  (correct)")
print(f"  True  ADL  predicted FALL : {fp}  (false alarm)")
print(f"  True  FALL predicted ADL  : {fn}  ← DANGEROUS (missed fall)")
print(f"  True  FALL predicted FALL : {tp}  (correct)")
print(f"\nMissed falls (false negatives): {fn}")
print(f"False alarms (false positives): {fp}")

=== TEST SET RESULTS ===

               precision    recall  f1-score   support

ADL (no fall)      0.979     0.970     0.974       471
         FALL      0.667     0.737     0.700        38

     accuracy                          0.953       509
    macro avg      0.823     0.854     0.837       509
 weighted avg      0.955     0.953     0.954       509

Confusion matrix:
  True  ADL  predicted ADL  : 457  (correct)
  True  ADL  predicted FALL : 14  (false alarm)
  True  FALL predicted ADL  : 10  ← DANGEROUS (missed fall)
  True  FALL predicted FALL : 28  (correct)

Missed falls (false negatives): 10
False alarms (false positives): 14


In [37]:
# ── CELL: Export for vision_engine.py ────────────────────────────────────────
# TorchScript trace — runs on CPU at inference, no GPU needed
model.load_state_dict(torch.load('/kaggle/working/fall_lstm_best.pt'))
model.cpu().eval()

example_input = torch.zeros(1, 30, 99)
traced        = torch.jit.trace(model, example_input)
traced.save('/kaggle/working/fall_lstm_traced.pt')

print("Files to download from /kaggle/working/:")
print("  fall_lstm_traced.pt  ← model for vision_engine.py")
print("  norm_mean.npy        ← normalization mean")
print("  norm_std.npy         ← normalization std")

# Verify traced model works
test_out = traced(example_input)
print(f"\n✅ Traced model output shape: {test_out.shape}")  # should be (1, 2)
print(f"   Probabilities: {torch.softmax(test_out, dim=1).detach().numpy()}")

Files to download from /kaggle/working/:
  fall_lstm_traced.pt  ← model for vision_engine.py
  norm_mean.npy        ← normalization mean
  norm_std.npy         ← normalization std

✅ Traced model output shape: torch.Size([1, 2])
   Probabilities: [[0.8351267  0.16487335]]
